In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 60.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=ac17cc8370a9928b0ddab7b3fc76d6f17c58f00fcfb88f39d12aa63c1efef858
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [3]:
# QUANTUM RANDOM BIT GENERATOR


def quantum_random_bit():

    qc = QuantumCircuit(1,1)

    # Put qubit into superposition
    qc.h(0)

    # Measure qubit
    qc.measure(0,0)

    simulator = BasicSimulator()

    compiled = transpile(qc, simulator)

    result = simulator.run(compiled, shots=1).result()

    counts = result.get_counts()

    measured_bit = int(list(counts.keys())[0])

    return measured_bit

In [4]:
# ALICE GENERATES RANDOM BITS AND BASES


n = 20

alice_bits = []
alice_bases = []

for i in range(n):

    alice_bits.append(quantum_random_bit())
    alice_bases.append(quantum_random_bit())

print("Alice Bits:")
print(alice_bits)

print("\nAlice Bases:")
print(alice_bases)

Alice Bits:
[1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0]

Alice Bases:
[1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0]


In [5]:
# ALICE ENCODES QUBITS
# Basis:
# 0 = Z basis
# 1 = X basis


qubits = []

for i in range(n):

    qc = QuantumCircuit(1,1)

    # Encode bit value
    if alice_bits[i] == 1:
        qc.x(0)

    # Encode basis
    if alice_bases[i] == 1:
        qc.h(0)

    qubits.append(qc)

In [6]:
# BOB CHOOSES RANDOM BASES AND MEASURES


bob_bases = []
bob_results = []

simulator = BasicSimulator()

for i in range(n):

    bob_basis = quantum_random_bit()

    bob_bases.append(bob_basis)

    qc = qubits[i].copy()

    # Measure in X basis
    if bob_basis == 1:
        qc.h(0)

    qc.measure(0,0)

    compiled = transpile(qc, simulator)

    result = simulator.run(compiled, shots=1).result()

    counts = result.get_counts()

    measured_bit = int(list(counts.keys())[0])

    bob_results.append(measured_bit)

print("Bob Bases:")
print(bob_bases)

print("\nBob Results:")
print(bob_results)

Bob Bases:
[1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]

Bob Results:
[1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0]


In [7]:
# ALICE AND BOB COMPARE BASES
# KEEP ONLY MATCHING BASES


shared_key_alice = []
shared_key_bob = []

for i in range(n):

    if alice_bases[i] == bob_bases[i]:

        shared_key_alice.append(alice_bits[i])
        shared_key_bob.append(bob_results[i])

print("Shared Key (Alice):")
print(shared_key_alice)

print("\nShared Key (Bob):")
print(shared_key_bob)

Shared Key (Alice):
[1, 1, 1, 0, 0, 1, 0, 1, 0, 0]

Shared Key (Bob):
[1, 1, 1, 0, 0, 1, 0, 1, 0, 0]


In [8]:
# ============================================================
# ERROR CHECKING
# ============================================================

errors = 0

for i in range(len(shared_key_alice)):

    if shared_key_alice[i] != shared_key_bob[i]:
        errors += 1

if len(shared_key_alice) > 0:
    error_rate = errors / len(shared_key_alice)
else:
    error_rate = 0

print("Error Rate:", error_rate)

if error_rate > 0.2:
    print("Attack Detected")
else:
    print("No Attacker Detected")

Error Rate: 0.0
No Attacker Detected
